## Ollama Basic Code snippets with Langchain

### Install Libraries

In [5]:
pip install -U langchain-ollama langchain-chroma langchain


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
embedding_model = "embeddinggemma:300m-qat-q4_0"
llm_model = "mistral:latest"

### Let's do a simple chat with ChatOllama

In [7]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# Initialize the chat model
model = ChatOllama(
    model=llm_model,
    temperature=0,
    validate_model_on_init=True
)

messages = [
    HumanMessage(content="Explain quantum computing in one sentence.")
]

response = model.invoke(messages)
print(response.content)


 Quantum computing is a type of computation that utilizes quantum-mechanical phenomena, such as superposition and entanglement, to perform operations on data using quantum bits (qubits), which can exist in multiple states at once, potentially solving certain complex problems much faster than classical computers.


### Using Embedding model

In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=embedding_model)
vector = embeddings.embed_query("This is a test document.")
print(vector)


In [8]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings

# 1. Initialize Local Ollama Embeddings and LLM
print("Initializing Ollama models...")
embeddings = OllamaEmbeddings(model=embedding_model)
llm = ChatOllama(model=llm_model, temperature=0)

# 2. Prepare Sample Documents (The Knowledge Base)
# In production, you'd use a text splitter / document loader here.
sample_docs = [
    Document(
        page_content="ReliableRAG is a specialized framework designed to eliminate LLM hallucinations using strict verification pipelines.",
        metadata={"source": "project_doc.txt"},
    ),
    Document(
        page_content="The core architecture of ReliableRAG uses a two-step validation: context relevance checking and response alignment.",
        metadata={"source": "architecture_doc.txt"},
    ),
    Document(
        page_content="Ollama runs LLMs locally on your machine using optimized quantization frameworks like llama.cpp.",
        metadata={"source": "ollama_doc.txt"},
    ),
]

# 3. Setup Local Chroma Vector Store (In-Memory for this example)
print("Embedding documents and indexing into Chroma DB...")
vector_store = Chroma.from_documents(
    documents=sample_docs,
    embedding=embeddings,
    persist_directory="./chroma_db",
)

# Create a retriever from the vector store
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# 4. Define the RAG Prompt Template
RAG_PROMPT = """
You are a helpful assistant. Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.

Context:
{context}

Question: {question}

Answer:
"""
prompt = ChatPromptTemplate.from_template(RAG_PROMPT)


# Helper function to format retrieved documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for docs in docs for doc in [docs])


# 5. Construct the LCEL (LangChain Expression Language) Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 6. Run a Query
query = "What is ReliableRAG and how does its architecture ensure validation?"
print(f"\n--- Querying: {query} ---")

response = rag_chain.invoke(query)
print("\nResponse:")
print(response)

Initializing Ollama models...
Embedding documents and indexing into Chroma DB...

--- Querying: What is ReliableRAG and how does its architecture ensure validation? ---

Response:
 ReliableRAG is a system that utilizes a two-step validation process for responses. The first step, context relevance checking, ensures the response is appropriate given the conversation context. The second step, response alignment, verifies the response aligns with the system's intended purpose or goal.


In [13]:
query = "What is Ollama"
print(f"\n--- Querying: {query} ---")

response = rag_chain.invoke(query)
print("\nResponse:")
print(response)


--- Querying: What is Ollama ---

Response:
 Ollama is a tool that runs Language Model (LLM) locally on your machine, utilizing optimized quantization frameworks such as llama.cpp for efficient processing.
